# CMU 11-768 Assignment 2 · Validator project walkthrough

In Assignment 1, we built the harness that lets a model use tools. Here an agent has already finished its run, and a validator has to decide from the task, the trajectory, and the saved `figure.png` which kinds of error the chart contains.

Open this notebook in Colab and run the cells in order. A CPU runtime is enough. Everything up to section 8 reads committed artifacts and predictions, so it makes no model calls and needs no API key. Section 9 is an optional live judge call that uses your own OpenAI key. The Docker sandbox and Harbor commands are in Markdown cells for use in Codespaces.

| Section | Follow in the repository | What you can run |
| --- | --- | --- |
| Part 1 | `validator/runner.py`, `validator/baseline.py`, `artifacts/`, `seed_labels.json` | Load a seed run, read its trajectory, and compare the baseline's verdict with the human label |
| Part 2 | `validator/solution.py`, `workflow/evaluate.py` | The file-based execution check on all 110 seed runs, both judge calls with a scripted judge, and MCC scoring of the committed predictions |
| Parts 3–4 | `tasks/`, `runs/`, `labels.json`, `output/`, `harbor/tasks/` | The 15 authored runs, the overlay experiment that was measured and switched off, and the Harbor value checks on a reference solution and two mutants |
| Live and Docker lab | `validator/model.py`, `workflow/sandbox_generate.py`, `sandbox/` | One optional judge call from Colab, then the Codespaces commands for agent generation and Harbor |

This project is an independent adaptation. The judge is `gpt-6.1-terra` and the three chart agents are hosted models, in place of the Qwen, Mistral and GLM checkpoints the course requires on Modal. The scores below describe this setup.

## 0 · Set up the source checkout

The examples pin the source, seed artifacts, and committed predictions to one commit. The notebook imports the validator and scorer into the Colab kernel. The repository's own tests run later in a separate `uv` environment.

In [ ]:
import importlib.util
import shutil
import subprocess
import sys
from pathlib import Path

SOURCE_COMMIT = "5dd09f26306de20576bcb1ba5cda1940a5c712ca"
repo = Path.cwd() / "assignment2_source"

if not repo.exists():
    subprocess.run(
        ["git", "clone", "--quiet", "https://github.com/xqzr2000/cmu-11-768-assignment-2.git", str(repo)],
        check=True,
    )
subprocess.run(["git", "-C", str(repo), "checkout", "--quiet", "--detach", SOURCE_COMMIT], check=True)

needed = {"dotenv": "python-dotenv", "openai": "openai", "pydantic": "pydantic", "tqdm": "tqdm",
          "PIL": "pillow", "matplotlib": "matplotlib", "pandas": "pandas", "pytest": "pytest"}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
sys.path.insert(0, str(repo))

import json

from validator.runner import runs

seed_runs = {run.run_id: run for run in runs()}
print(f"{len(seed_runs)} seed runs loaded from {repo.name}/artifacts")
assert len(seed_runs) == 110

## 1 · Read one run the way the validator does

### 1.1 · The run record

Each directory under `artifacts/` holds a `result.json` and, when the agent saved one, a `figure.png`. `Run.load` turns that into the object every validator receives: the task instructions, the chat messages of the trajectory, the figure path, and any input files. `transcript` is the compact text version of the trajectory that `validator/solution.py` puts in the judge prompt. It keeps the agent's commands and clips long tool output.

Seed run `021` asks for a box plot. Read the instructions and the agent's script before looking at the labels in the next cell.

In [ ]:
from IPython.display import Image as show_image, display

from validator.solution import image_facts, transcript

run = seed_runs["021"]
print(run.instructions)
print("\nInput files:", list(run.inputs) or "none", "| messages:", len(run.messages))
print("Measured from the PNG:", image_facts(run.figure))
print("\n--- transcript (first 1,400 characters) ---")
print(transcript(run.messages)[:1400])
display(show_image(filename=str(run.figure), width=560))

### 1.2 · Human label, baseline verdict, final verdict

A label or a prediction is a list of errors, each with one of four families and a sentence of evidence. `execution_failure` means there is no usable figure. `wrong_data` is about which values are plotted, `wrong_chart` about how they are drawn, and `hard_to_read` about legibility of what was drawn.

The human label for `021` has one error. The starter baseline in `validator/baseline.py` asks the judge one yes/no question per family, and here it reports two. Read the baseline's `wrong_data` evidence closely.

In [ ]:
def families_by_run(path, key):
    return {
        item["run_id"]: {error["family"]: error["evidence"] for error in item["errors"]}
        for item in json.loads(path.read_text())[key]
    }


seed_labels = families_by_run(repo / "seed_labels.json", "labels")
baseline = families_by_run(repo / "output" / "baseline_predictions.json", "predictions")
final = families_by_run(repo / "output" / "final_seed_predictions.json", "predictions")

for name, verdicts in (("Human label", seed_labels), ("Baseline", baseline), ("Final validator", final)):
    print(name)
    for family, evidence in verdicts["021"].items():
        print(f"  {family}: {evidence}")
assert set(seed_labels["021"]) == set(final["021"]) == {"wrong_chart"}
assert set(baseline["021"]) == {"wrong_chart", "wrong_data"}

The baseline's `wrong_data` evidence says the data matches the request and then cites the missing axis label, which is a design problem. Section 5 counts how often this happened across the seed set. Try `"090"` in place of `"021"` in both cells for a second case.

## 2 · Decide `execution_failure` from the file

The baseline judged execution from the trajectory. A run that hit its step limit after saving a figure looked like a crash, and a run with a tidy transcript and no figure looked fine. `judge_execution` in `validator/solution.py` looks only at the artifact. A `figure.png` that is missing, has another format, or fails to decode is an execution failure, and the result is final without a model call.

Run that check on all 110 seed runs and compare it with the human labels. Then list the seed runs where the baseline got this family wrong.

In [ ]:
from validator.solution import judge_execution

gate = {run_id: bool(judge_execution(run)) for run_id, run in seed_runs.items()}
truth = {run_id: "execution_failure" in seed_labels[run_id] for run_id in seed_runs}
print("Flagged by the file check:", sum(gate.values()), "| labelled execution_failure:", sum(truth.values()))
assert gate == truth

predicted = {run_id: "execution_failure" in baseline[run_id] for run_id in seed_runs}
false_alarms = [run_id for run_id in seed_runs if predicted[run_id] and not truth[run_id]]
missed = [run_id for run_id in seed_runs if truth[run_id] and not predicted[run_id]]
print("Baseline reported a failure although a PNG exists:", false_alarms)
print("Baseline missed a run with no PNG:", missed)
exit_status = (seed_runs[false_alarms[0]].messages[-1].get("extra") or {}).get("exit_status")
print(f"Run {false_alarms[0]} ended with status {exit_status!r} and has", image_facts(seed_runs[false_alarms[0]].figure))

The 12 labelled execution failures are the 12 runs without a figure, so the file check scores this family exactly on the seed set. The check is also strict about damaged files. Make two broken copies of a good figure and pass them through the same function.

In [ ]:
import tempfile
from dataclasses import replace

from PIL import Image

scratch = Path(tempfile.mkdtemp(prefix="walkthrough_"))
good = seed_runs["021"].figure

truncated = scratch / "truncated" / "figure.png"
truncated.parent.mkdir()
truncated.write_bytes(good.read_bytes()[: good.stat().st_size // 2])

renamed_jpeg = scratch / "jpeg" / "figure.png"
renamed_jpeg.parent.mkdir()
with Image.open(good) as image:
    image.convert("RGB").save(renamed_jpeg, format="JPEG")

for label, figure in (("original", good), ("truncated", truncated),
                      ("JPEG named figure.png", renamed_jpeg), ("no file", None)):
    errors = judge_execution(replace(seed_runs["021"], figure=figure))
    print(f"{label:22s} -> {errors[0].evidence if errors else 'valid PNG, continue to the judge calls'}")

## 3 · Follow the two judge calls with a scripted judge

For a run with a valid figure, `validate` makes two calls to the vision judge.

1. **Requirement checklist.** The judge lists every requirement in the instruction, tags each one `data`, `design` or `legibility`, and marks it `met`, `violated` or `unclear`. `wrong_data` is reported only if a `data` item is violated, and `wrong_chart` only if a `design` item is violated.
2. **Readability.** A separate call looks at the image for clipped, overlapping, hidden or low-contrast content. Only issues rated `major` count as `hard_to_read`.

Here a scripted function stands in for `validator.model.complete`, so you can see the exact request and how a reply becomes a prediction. The reply text below is written by hand for run `021`. The parsing and the family mapping are the real code.

In [ ]:
from unittest.mock import patch

import validator.solution as solution


def scripted_judge(*replies):
    calls = []

    def complete(messages, **options):
        calls.append({"messages": messages, "options": options})
        reply = replies[len(calls) - 1]
        return {"choices": [{"message": {"content": json.dumps(reply)}}]}

    return complete, calls


checklist_reply = {"requirements": [
    {"requirement": "One box per service in the given order", "kind": "data",
     "status": "met", "evidence": "auth, catalog, search, checkout from left to right"},
    {"requirement": "Label the x axis", "kind": "design",
     "status": "violated", "evidence": "tick labels are present but the axis has no label"},
]}
readability_reply = {"issues": [
    {"category": "squeezed_layout", "where": "catalog box", "severity": "minor",
     "description": "the box is short but its median is readable"},
]}

fake, calls = scripted_judge(checklist_reply, readability_reply)
with patch.object(solution, "complete", fake):
    errors = solution.validate(seed_runs["021"])

for number, call in enumerate(calls, start=1):
    parts = call["messages"][0]["content"]
    schema = call["options"]["response_format"]["json_schema"]["name"]
    print(f"Call {number}: schema={schema}, text parts={sum(p['type'] == 'text' for p in parts)}, "
          f"images={sum(p['type'] == 'image_url' for p in parts)}")
print("\n--- start of the checklist prompt ---")
print(calls[0]["messages"][0]["content"][0]["text"][:900])
print("\nPrediction:", [(error.family.value, error.evidence) for error in errors])
assert [error.family.value for error in errors] == ["wrong_chart"]

Change the first item's `status` to `"violated"` and rerun. The prediction gains `wrong_data`, with that item's evidence. Change the readability issue to `"major"` and it gains `hard_to_read`. A missing axis label can no longer be filed under `wrong_data`, because the family comes from the item's `kind`.

### 3.1 · The Part 4 `legibility` kind and the overlay flag

On the authored runs, the judge filed the instruction that labels must not overlap as a violated `design` item, which produced a `wrong_chart` false positive. Part 4 added a third kind, `legibility`. The checklist lists those items and they never count toward `wrong_chart`. The readability call decides them.

Part 4 also tried an overlay inventory. With `OVERLAY_INVENTORY` on, the readability call receives the whole image plus four enlarged quadrants, and the judge must list every legend or text box and what lies under it. The next cell runs both settings on the authored run `sensor-daily-median__luna`, whose legend covers part of a shaded band.

In [ ]:
authored_runs = {run.run_id: run for run in runs(repo / "runs")}
sensor = authored_runs["sensor-daily-median__luna"]

checklist = {"requirements": [
    {"requirement": "Legend with the three named entries", "kind": "design",
     "status": "met", "evidence": "all three entries are present"},
    {"requirement": "Every label fully visible and not overlapping", "kind": "legibility",
     "status": "violated", "evidence": "the legend sits on the S2 band"},
]}
for flag in (False, True):
    readability = {"issues": []}
    if flag:
        readability["overlays"] = [{"element": "legend", "covers": "data", "hides_information": True,
                                    "detail": "upper edge of the S2 daily range near 06-02"}]
    fake, calls = scripted_judge(checklist, readability)
    with patch.object(solution, "complete", fake), patch.object(solution, "OVERLAY_INVENTORY", flag):
        errors = solution.validate(sensor)
    images = sum(part["type"] == "image_url" for part in calls[1]["messages"][0]["content"])
    print(f"OVERLAY_INVENTORY={flag}: readability call sends {images} image(s) ->",
          [error.family.value for error in errors] or "no errors")
print("Shipped setting: OVERLAY_INVENTORY =", solution.OVERLAY_INVENTORY)
assert solution.OVERLAY_INVENTORY is False

In both settings the violated `legibility` item stays out of `wrong_chart`. The scripted replies show the mechanism only. Section 6 uses the recorded predictions to show what the real judge did with each setting and why the flag was left off.

## 4 · Score predictions with MCC

`workflow/evaluate.py` scores each family as its own yes/no classification over the runs. With true and false positives and negatives $TP, TN, FP, FN$, the Matthews correlation coefficient is

$$\mathrm{MCC} = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

It is 1 for perfect agreement, 0 for a validator that carries no information, and it stays informative when one class is rare. The macro-MCC is the plain mean over the four families. A run labelled `execution_failure` is scored on that family alone and left out of the other three, because a figure that does not exist says nothing about data or design. That is why `wrong_data` below has 98 scored runs.

Compute one family by hand, then score the three committed seed prediction files.

In [ ]:
from math import sqrt

import matplotlib.pyplot as plt

from validator.prediction import ERROR_FAMILIES
from workflow.evaluate import score

tp = tn = fp = fn = 0
for run_id, label in seed_labels.items():
    if "execution_failure" in label:
        continue
    actual, said = "wrong_data" in label, "wrong_data" in final[run_id]
    tp += actual and said
    tn += not actual and not said
    fp += said and not actual
    fn += actual and not said
by_hand = (tp * tn - fp * fn) / sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
print(f"wrong_data by hand: TP={tp} TN={tn} FP={fp} FN={fn} over {tp + tn + fp + fn} runs, MCC={by_hand:.4f}")

seed_stages = {
    "Baseline": "baseline_predictions.json",
    "Part 2": "pre_iteration_seed_predictions.json",
    "Final": "final_seed_predictions.json",
}
seed_reports = {
    stage: score(repo / "seed_labels.json", repo / "output" / filename)
    for stage, filename in seed_stages.items()
}
assert abs(seed_reports["Final"]["wrong_data"]["mcc"] - by_hand) < 1e-12

fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
columns = [*ERROR_FAMILIES, "macro_mcc"]
for offset, (stage, report) in enumerate(seed_reports.items()):
    values = [report[name]["mcc"] if name != "macro_mcc" else report[name] for name in columns]
    bars = ax.bar([x + (offset - 1) * 0.27 for x in range(len(columns))], values, width=0.27, label=stage)
    ax.bar_label(bars, fmt="%.2f", fontsize=7)
    print(f"{stage:9s} macro-MCC = {report['macro_mcc']:.4f}")
ax.set_xticks(range(len(columns)), [name.replace("macro_mcc", "macro-MCC") for name in columns])
ax.set(ylabel="MCC on the 110 seed runs", ylim=(0, 1.1))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=3, frameon=False)
plt.show()

assert [round(report["macro_mcc"], 4) for report in seed_reports.values()] == [0.5235, 0.8747, 0.8726]

The final macro-MCC is 0.0021 below the Part 2 pass. The final file has one fewer `wrong_chart` miss and one more `wrong_data` miss than the Part 2 file. The project notes record that two passes of the same Part 2 validator differed by about 0.01 at temperature 0, so the Part 4 change shows no measurable effect on the seed set in either direction.

## 5 · Where the baseline's `wrong_data` false positives came from

`wrong_data` had the lowest baseline MCC. Take every seed run where the baseline reported `wrong_data` and the human label has none, and look at what the human label says instead. Repeat for the final validator.

In [ ]:
def wrong_data_false_positives(predictions):
    return [run_id for run_id, label in seed_labels.items()
            if "execution_failure" not in label
            and "wrong_data" in predictions[run_id] and "wrong_data" not in label]


for name, predictions in (("Baseline", baseline), ("Final validator", final)):
    flagged = wrong_data_false_positives(predictions)
    design = [run_id for run_id in flagged if "wrong_chart" in seed_labels[run_id]]
    print(f"{name}: {len(flagged)} wrong_data false positives, "
          f"{len(design)} of them on runs the humans labelled wrong_chart")

run_id = "090"
print(f"\nRun {run_id}")
print("  human   :", seed_labels[run_id])
print("  baseline:", baseline[run_id]["wrong_data"])
print("  final   :", sorted(final[run_id]))
assert len(wrong_data_false_positives(baseline)) == 46
assert len(wrong_data_false_positives(final)) == 9

All 46 baseline false positives are runs with a real design error. Asked whether the data differs in any way, the judge answered yes for any problem it saw. The checklist reduces this to nine, and those nine are also runs with a design error. What remains is the judge tagging a design requirement as `data`.

To read every disagreement with its evidence, run `scripts/disagreements.py` on a prediction file. It prints each false positive and false negative per family.

## 6 · The authored tasks and the Part 4 trade-off

Part 3 added five tasks in two classes that could expose a validator: aggregation over large CSV inputs, and fine visual detail. Each task was run once by three hosted agents inside the Docker sandbox, giving the 15 runs in `runs/`. The grid shows every figure with its human-reviewed label.

In [ ]:
authored_labels = families_by_run(repo / "labels.json", "labels")
task_ids = sorted({run.task_id for run in authored_runs.values()})
agents = ("deepseek", "luna", "glm")

fig, axes = plt.subplots(len(task_ids), len(agents), figsize=(15, 16), layout="constrained")
for row, task_id in enumerate(task_ids):
    for column, agent in enumerate(agents):
        run_id = f"{task_id}__{agent}"
        label = sorted(authored_labels[run_id])
        ax = axes[row, column]
        ax.imshow(plt.imread(authored_runs[run_id].figure))
        ax.set_title(f"{run_id}\n{', '.join(label) or 'acceptable'}", fontsize=9,
                     color="firebrick" if label else "black")
        ax.axis("off")
plt.show()

positives = {family: sum(family in label for label in authored_labels.values()) for family in ERROR_FAMILIES}
print("Positive labels among the 15 runs:", positives)
assert positives == {"execution_failure": 0, "wrong_data": 0, "wrong_chart": 0, "hard_to_read": 5}

Every agent plotted the right values with the requested design. The five errors are all `hard_to_read`: two department-budget charts with clipped or colliding names, and all three sensor charts, where the legend covers the upper edge of the S2 band. With no positive example in three families, the scorer reports their MCC as 0 by convention, so the authored macro-MCC has a ceiling of 0.25.

The table compares four validator versions on both sets. `Part 4a` turned on the overlay inventory, `Part 4b` added the rule that an overlay must hide information a reader needs, and `Final` keeps only the `legibility` kind.

In [ ]:
versions = {
    "Part 2": ("pre_iteration_seed", "pre_iteration_authored"),
    "Part 4a": ("part4a_seed", "part4a_authored"),
    "Part 4b": ("part4b_seed", "part4b_authored"),
    "Final": ("final_seed", "authored"),
}
print(f"{'':9s}{'--- 110 seed runs ---':>28s}{'--- 15 authored runs ---':>34s}")
print(f"{'':9s}{'macro':>8s}{'h2r MCC':>9s}{'FP':>4s}{'FN':>4s}"
      f"{'macro':>12s}{'h2r MCC':>9s}{'FP':>4s}{'FN':>4s}{'chart FP':>10s}")
part4 = {}
for version, (seed_file, authored_file) in versions.items():
    on_seeds = score(repo / "seed_labels.json", repo / "output" / f"{seed_file}_predictions.json")
    on_authored = score(repo / "labels.json", repo / "output" / f"{authored_file}_predictions.json")
    part4[version] = (on_seeds, on_authored)
    s, a = on_seeds["hard_to_read"], on_authored["hard_to_read"]
    print(f"{version:9s}{on_seeds['macro_mcc']:8.4f}{s['mcc']:9.4f}{s['fp']:4d}{s['fn']:4d}"
          f"{on_authored['macro_mcc']:12.4f}{a['mcc']:9.4f}{a['fp']:4d}{a['fn']:4d}"
          f"{on_authored['wrong_chart']['fp']:10d}")

assert part4["Part 4a"][1]["hard_to_read"]["fn"] == 0 and part4["Part 4a"][0]["hard_to_read"]["fp"] == 12
assert part4["Final"][1]["wrong_chart"]["fp"] == 0 and part4["Final"][1]["hard_to_read"]["fn"] == 3

The overlay inventory caught all five authored readability errors. On the seed runs it raised `hard_to_read` false positives from 2 to 12, because value labels drawn on their own bars and annotation arrows touching their points were reported as covering data. The calibrated rule brought that to 7, still worse than Part 2. The seed set is seven times larger and more varied than the authored set, so the final validator ships with the flag off and still misses the three sensor legends. The `legibility` kind removed the one authored `wrong_chart` false positive.

## 7 · Harbor verifiers: read the figure's artist tree

Each authored task is also packaged under `harbor/tasks/` with a deterministic verifier. The verifier re-runs the solution's `plot.py` with a `sitecustomize.py` hook on `PYTHONPATH`. Python imports that file at start-up, and it patches `Figure.savefig` to write a JSON manifest of the figure next to the PNG: bar heights, line data, tick labels, colours, axis text. The test functions then compare the manifest with an answer key, so no pixels are read.

The real verifier runs in Docker under `/app`. This cell uses the same hook, answer keys and assertions on Colab. It renders the reference solution and the `sum_units` mutant, which sums units in place of revenue, and calls the task's own manifest tests on each.

In [ ]:
import inspect
import os
import re


def render(task_id, solve_script):
    """Run the plot.py embedded in a solve.sh with the verifier's savefig hook."""
    task_dir = repo / "harbor" / "tasks" / task_id
    work = Path(tempfile.mkdtemp(prefix=f"{task_id}_"))
    for source in (task_dir / "environment").iterdir():
        if source.name != "Dockerfile":
            shutil.copy2(source, work / source.name)
    plot_source = re.search(r"<<'PLOT_EOF'\n(.*?)\nPLOT_EOF", solve_script.read_text(), re.S).group(1)
    (work / "plot.py").write_text(plot_source + "\n")
    environment = dict(os.environ, PYTHONPATH=str(task_dir / "tests"), MPLBACKEND="Agg")
    subprocess.run([sys.executable, "plot.py"], cwd=work, env=environment, check=True)
    return work, json.loads((work / "figure.manifest.json").read_text())


def manifest_checks(task_id, manifest):
    """Call the task's own test functions that need only the manifest."""
    tests = repo / "harbor" / "tasks" / task_id / "tests"
    if str(tests) not in sys.path:
        sys.path.append(str(tests))
    spec = importlib.util.spec_from_file_location(f"{task_id.replace('-', '_')}_tests", tests / "test_state.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    results = {}
    for name, function in vars(module).items():
        if name.startswith("test_s2_") and list(inspect.signature(function).parameters) == ["manifest"]:
            try:
                function(manifest)
                results[name] = "passed"
            except AssertionError as error:
                results[name] = f"FAILED: {str(error)[:110]}"
    return module, results


task_id = "quarterly-revenue-by-region"
task_dir = repo / "harbor" / "tasks" / task_id
outcomes = {}
for variant, script in (("reference", task_dir / "solution" / "solve.sh"),
                        ("sum_units mutant", task_dir / "mutants" / "sum_units" / "solve.sh")):
    work, manifest = render(task_id, script)
    module, outcomes[variant] = manifest_checks(task_id, manifest)
    east = next(c for c in manifest["axes"][0]["containers"] if c.get("label") == "East")
    print(f"{variant}: East bars from the manifest = {[round(value, 1) for value in east['values']]}")
    for name, outcome in outcomes[variant].items():
        print(f"   {name}: {outcome}")
print("Answer key for East:", module.KEY["East"])
assert set(outcomes["reference"].values()) == {"passed"}
assert outcomes["sum_units mutant"]["test_s2_sixteen_vertical_bars_with_the_right_heights"].startswith("FAILED")

The mutant's chart looks plausible, and the bar-height test rejects it at the first value. In the recorded Docker run this mutant scored 0.0.

The second mutant shows the limit of this approach. `unrotated_labels` draws the 30 department names horizontally. Run the same manifest tests on it and then look at both figures.

In [ ]:
task_id = "dense-department-budgets"
task_dir = repo / "harbor" / "tasks" / task_id
fig, axes = plt.subplots(2, 1, figsize=(12, 11), layout="constrained")
for ax, (variant, script) in zip(axes, (
        ("reference", task_dir / "solution" / "solve.sh"),
        ("unrotated_labels mutant", task_dir / "mutants" / "unrotated_labels" / "solve.sh"))):
    work, manifest = render(task_id, script)
    _, results = manifest_checks(task_id, manifest)
    passed = sum(outcome == "passed" for outcome in results.values())
    assert passed == len(results) > 0
    ax.imshow(plt.imread(work / "figure.png"))
    ax.set_title(f"{variant}: {passed}/{len(results)} manifest tests passed")
    ax.axis("off")
plt.show()
print((repo / "output" / "check_submission.txt").read_text())

Both versions pass, since the manifest holds the same names, order and values. No test measures rotation, overlap or clipping. The recorded `check-submission` output above shows the same result from the real Harbor run: the five reference solutions score 1.0, `sum_units` scores 0.0, and `unrotated_labels` is accepted with 1.0.

Exact value checks and a vision judge fail in different places. The verifier is exact on numbers and blind to legibility. The judge sees legibility problems and, with large inputs, cannot verify a sum.

## 8 · Run the repository's offline checks

The repository pins its dependencies in `uv.lock`, including the Assignment 1 sandbox runtime as a Git dependency. `uv sync` builds that environment next to the notebook kernel. The first run downloads the packages and can take a few minutes.

The checks run the validator's offline unit tests, validate the authored task descriptors and labels, and re-score the final predictions through the command-line workflow. None of them calls a model. The two `live mode requires` lines in the output come from tests that check a live request is refused without its confirmation flags.

In [ ]:
def in_repo(*command, **options):
    return subprocess.run(["uv", "run", "--locked", "--no-sync", *command], cwd=repo, check=True, **options)


subprocess.run(["uv", "sync", "--locked", "--python", "3.12", "--quiet"], cwd=repo, check=True)
in_repo("python", "scripts/offline_preflight.py")
in_repo("python", "-m", "unittest", "discover", "-s", "tests", "-p", "test_validator_*_offline.py")
in_repo("python", "-m", "workflow", "validate-tasks")
in_repo("python", "-m", "workflow", "validate-labels")
in_repo("python", "-m", "workflow", "score", "--labels", "seed_labels.json",
        "--predictions", "output/final_seed_predictions.json")
in_repo("python", "-m", "workflow", "score", "--predictions", "output/authored_predictions.json")

The last command prints an insufficient-support warning for three families. That is the scorer's convention from section 6 for a family with no positive labels.

## 9 · Optional live judge call

This cell sends one seed run to the hosted judge with the settings of the recorded experiment: `gpt-6.1-terra` through the OpenAI Chat Completions API, reasoning effort `none`, temperature 0 and the flex service tier. A run with a valid figure costs two requests that each include the image, and is billed to your OpenAI account. Model names and tiers depend on what your account offers when you run it.

Add `OPENAI_API_KEY` under the key icon in Colab's left sidebar and give this notebook access. Run section 8 first, because the call uses that `uv` environment. Then set `RUN_LIVE_JUDGE = True`. The judge is a model, so a rerun can differ from the committed prediction.

In [ ]:
RUN_LIVE_JUDGE = False
LIVE_RUN_ID = "021"

if RUN_LIVE_JUDGE:
    try:
        from google.colab import userdata

        api_key = userdata.get("OPENAI_API_KEY")
    except ImportError:
        api_key = os.environ.get("OPENAI_API_KEY")
    assert api_key, "Set OPENAI_API_KEY as a Colab secret or an environment variable."

    live = Path(tempfile.mkdtemp(prefix="live_judge_"))
    shutil.copytree(repo / "artifacts" / LIVE_RUN_ID, live / "artifacts" / LIVE_RUN_ID)
    judge_environment = dict(
        os.environ,
        OPENAI_API_KEY=api_key,
        VALIDATOR_BASE_URL="https://api.openai.com/v1",
        VALIDATOR_MODEL="gpt-6.1-terra",
        VALIDATOR_REASONING_EFFORT="none",
        VALIDATOR_SERVICE_TIER="flex",
    )
    judge_environment.pop("VALIDATOR_API_KEY", None)
    in_repo("python", "-m", "validator.runner",
            "--artifacts", str(live / "artifacts"), "--artifact-root", str(repo),
            "--out", str(live / "predictions.json"), env=judge_environment)

    live_prediction = families_by_run(live / "predictions.json", "predictions")[LIVE_RUN_ID]
    for name, verdict in (("Live judge", live_prediction), ("Committed final", final[LIVE_RUN_ID]),
                          ("Human label", seed_labels[LIVE_RUN_ID])):
        print(f"{name:16s}", sorted(verdict) or "acceptable")
    for family, evidence in live_prediction.items():
        print(f"\n{family}: {evidence}")
else:
    print("To call the hosted judge, add an OPENAI_API_KEY secret and set RUN_LIVE_JUDGE = True.")

## 10 · Docker lab in Codespaces

Agent generation and the real Harbor runs need Docker, which a Colab runtime does not provide. Use a Codespace on the repository, or any machine with Docker, Python 3.12 and `uv`. The commands run from the repository root. The model calls are billed by each provider. The [README](https://github.com/xqzr2000/cmu-11-768-assignment-2/blob/main/README.md) has the full reproduction notes and describes the sandbox boundary.

| Step | Entry point | What runs where |
| --- | --- | --- |
| Generate agent runs | `workflow/sandbox_generate.py` | The host holds the API keys and calls the model. Each run gets a fresh container with networking off that executes the agent's bash commands. Only `figure.png` is copied back |
| Judge the runs | `validator/runner.py` | The host sends the task, transcript, inputs and figure to the hosted judge |
| Score | `workflow score` | Offline comparison with human labels |
| Harbor oracles and mutants | `workflow check-submission` | Each packaged task builds its own image and runs its verifier with no network |

### 10.1 · Generate one agent run in the sandbox

Set `DEEPSEEK_API_KEY`, `OPENAI_API_KEY` and `ZAI_API_KEY` as Codespaces secrets or in an ignored `.env` file. Build the plotting image once. Write the probe to a new directory so the 15 committed runs and their labels stay as they are.

```bash
uv sync --locked
docker build -t hw2-chart:1 sandbox/

uv run python -m workflow.sandbox_generate \
  --only styled-decay-curves --models glm --artifacts runs_probe
```

The agent keys are `deepseek`, `luna` and `glm`. A run stops after 20 model steps, 120 seconds per command, or 15 minutes in total. Check `runs_probe/styled-decay-curves__glm/` for `result.json` and `figure.png`. Leave out `--only` and `--models` to run every task with every agent.

### 10.2 · Judge and score your runs

For the Terra judge, set `VALIDATOR_BASE_URL=https://api.openai.com/v1`, `VALIDATOR_MODEL=gpt-6.1-terra`, `VALIDATOR_REASONING_EFFORT=none` and `VALIDATOR_SERVICE_TIER=flex` in `.env`, and leave `VALIDATOR_API_KEY` unset so the client uses `OPENAI_API_KEY`. Write predictions to new files so the committed ones stay intact.

```bash
uv run python -m validator.runner --artifacts runs_probe --out output/my_probe_predictions.json

uv run python -m validator.runner --artifacts artifacts \
  --out output/my_seed_predictions.json --workers 4
uv run python -m workflow score --labels seed_labels.json \
  --predictions output/my_seed_predictions.json
uv run python scripts/disagreements.py output/my_seed_predictions.json
```

A new agent run has no human label yet. Look at its figure and write the label before you read the validator's prediction, then add it to a label file and score. If the runner stops with failed runs, fix the cause and repeat the command with `--resume` to judge only the missing ones.

### 10.3 · Run the Harbor oracles and mutants

```bash
uv run harbor run -p harbor/tasks/quarterly-revenue-by-region -a oracle \
  --job-name quarterly-revenue-by-region-oracle

uv run python -m workflow check-submission
```

The first command runs one reference solution through its verifier and should report a reward of 1.0. `check-submission` validates the whole submission and reruns all five oracles and both mutants. Compare its output with [`output/check_submission.txt`](https://github.com/xqzr2000/cmu-11-768-assignment-2/blob/main/output/check_submission.txt).

### 10.4 · What to look for

| What we saw | How we checked it | What the repository does |
| --- | --- | --- |
| The baseline reported crashes for runs that had saved a figure, and accepted runs with none | Compared `execution_failure` predictions with whether `figure.png` exists | `judge_execution` decides from the file and makes no model call |
| 46 `wrong_data` false positives on runs with a design error | Read the baseline's evidence next to the human label | One checklist call tags each requirement `data`, `design` or `legibility` |
| Two passes of the same validator differed by about 0.01 macro-MCC at temperature 0 | Scored both Part 2 passes | Differences of that size are treated as noise |
| The overlay inventory fixed the authored misses and added seed false positives | Scored the seed and authored sets after each change | `OVERLAY_INVENTORY = False` in the final validator |
| The `unrotated_labels` mutant scored 1.0 in Harbor | Looked at the figure the verifier accepted | Recorded as a verifier blind spot in the report |
| Newer OpenAI models reject `max_tokens` or a non-default `temperature` | The API returns a 400 that names the parameter | `validator/model.py` renames or drops the parameter and retries |

Follow the [final report](https://github.com/xqzr2000/cmu-11-768-assignment-2/blob/main/report.pdf) for the full error analysis, the [iteration notes](https://github.com/xqzr2000/cmu-11-768-assignment-2/blob/main/docs/report-notes.md) for the measurements behind each decision, and the [learning log](https://github.com/xqzr2000/cmu-11-768-assignment-2/blob/main/docs/Learning-Progress.md) for the session-by-session history.